# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset DOI: {getattr(metadata, 'identifier', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', None)}")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their field @ids
record_sets = dataset.record_sets  # List of RecordSet objects

print("Available Record Sets:")
rs_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    rs_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for f in rs.fields:
            colinfo = ''
            if hasattr(f, 'column') and f.column:
                cols = [c if isinstance(c, str) else (c.get('id', '')) for c in (f.column if isinstance(f.column, list) else [f.column])]
                colinfo = f"(columns: {', '.join(cols)})"
            print(f"      - Field name: {f.name}, @id: {f.id} {colinfo}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set as DataFrames using record set @ids

# Use the list of record set ids discovered above
dataframes = {}
for rs_id in rs_ids:
    print(f"Loading records from recordSet @id: {rs_id}")
    # Use try/except in case some recordSets have no records yet
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  [No record data available]")
    except Exception as exc:
        print(f"  [Error loading: {exc}]")
print("")
# Preview the first available DataFrame
if dataframes:
    first_rs = next(iter(dataframes.keys()))
    print(f"#### First 5 rows of DataFrame for recordSet @id: {first_rs}")
    display(dataframes[first_rs].head())
else:
    print("No dataframes could be constructed from the record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes steps such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

_**Note**: All columns, fields, and record set names are referenced by their `@id` fields for clarity and reproducibility._

In [ ]:
import numpy as np

# Choose a record set with data.
if dataframes:
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]

    # Try to detect a numeric column by pandas dtype
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if numeric_cols.any():
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Set a threshold (use mean or fixed value if known)
        threshold = float(df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical column, if available
        cat_cols = df.select_dtypes(include=[object]).columns
        group_field_id = None
        for col in cat_cols:
            if col != numeric_field_id and len(df[col].unique()) > 1 and len(df[col].unique()) < len(df)/2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print('No numeric field available for EDA in the first record set.')
else:
    print('No record set DataFrame loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we provide examples using matplotlib and seaborn. Adapt `field_id` as appropriate for your dataset (always reference fields by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Use the DataFrame and field ids chosen above
    if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()

        if 'group_field_id' in locals() and group_field_id:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print('No filtered numeric data available for visualization.')
else:
    print('No data loaded for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains ordered logistic regression outputs and household survey responses, with metadata available via the Croissant schema.
- Data can be loaded and processed by referencing all entities using their `@id` fields.
- Filtering and normalization steps can aid in understanding numeric trends, and visualizations can highlight patterns across groups identified by field `@id`s.
- This notebook can serve as a starting point for further statistical analysis, domain investigation, or integration into downstream policy or research workflows.